In [1]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Payments Profiling").getOrCreate()

c:\Users\LENOVO\Desktop\AdvancedPySparkPipeline\.venv\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


In [2]:
payments = spark.read.parquet("../data/bronze/payments")

In [3]:
payments.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- payment_sequential: string (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- payment_installments: string (nullable = true)
 |-- payment_value: string (nullable = true)



In [4]:
payments.count()

103886

In [5]:
payments.show(10, truncate=False)

+--------------------------------+------------------+------------+--------------------+-------------+
|order_id                        |payment_sequential|payment_type|payment_installments|payment_value|
+--------------------------------+------------------+------------+--------------------+-------------+
|b81ef226f3fe1789b1e8b2acac839d17|1                 |credit_card |8                   |99.33        |
|a9810da82917af2d9aefd1278f1dcfa0|1                 |credit_card |1                   |24.39        |
|25e8ea4e93396b6fa0d3dd708e76c1bd|1                 |credit_card |1                   |65.71        |
|ba78997921bbcdc1373bb41e913ab953|1                 |credit_card |8                   |107.78       |
|42fdf880ba16b47b59251dd489d4441a|1                 |credit_card |2                   |128.45       |
|298fcdf1f73eb413e4d26d01b25bc1cd|1                 |credit_card |2                   |96.12        |
|771ee386b001f06208a7419e4fc1bbd7|1                 |credit_card |1               

In [6]:
from pyspark.sql.functions import col, count, when
payments.select([
    count(when(col(c).isNull(),c)).alias(c)
    for c in payments.columns
]).show(vertical=True)

-RECORD 0-------------------
 order_id             | 0   
 payment_sequential   | 0   
 payment_type         | 0   
 payment_installments | 0   
 payment_value        | 0   



In [7]:
duplicate_payments= payments.groupBy("order_id", "payment_sequential").count().filter(col("count")>1)
duplicate_payments.show()

+--------+------------------+-----+
|order_id|payment_sequential|count|
+--------+------------------+-----+
+--------+------------------+-----+



In [8]:
payments.select("payment_type").distinct().show()

+------------+
|payment_type|
+------------+
|      boleto|
| not_defined|
| credit_card|
|     voucher|
|  debit_card|
+------------+

